# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimanshahid800/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [3]:
!git clone https://github.com/aimanshahid800/flyrank-ml-internship.git
%cd flyrank-ml-internship

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 149, done.
remote: Counting objects: 100% (149/149), done.
remote: Compressing objects: 100% (105/105), done.
remote: Total 149 (delta 57), reused 92 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (149/149), 1.87 MiB | 17.10 MiB/s, done.
Resolving deltas: 100% (57/57), done.
/content/flyrank-ml-internship


In [4]:
from google.colab import userdata
import duckdb

hf_token = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE SECRET hf_secret (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

print("DuckDB + HF token ready")

DuckDB + HF token ready


In [5]:
# Setup: reload data + retrain model (same as w05_model.ipynb)
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

model_df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
model_df["is_declining"] = (model_df["trend_direction"] == "down").astype(int)

features = ["content_age_days", "impressions_last_30d", "avg_position", "ctr"]
model_df = model_df.dropna(subset=features + ["is_declining"])

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(model_df, groups=model_df["content_id"]))
train_df = model_df.iloc[train_idx]

X_train, y_train = train_df[features], train_df["is_declining"]

rf = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, class_weight="balanced")
rf.fit(X_train, y_train)

print("Model retrained, ready for scoring.")

Model retrained, ready for scoring.


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 1: Ranked actions + reason codes

# Score ALL pages (not just test set) with the trained model
model_df["decline_prob"] = rf.predict_proba(model_df[features])[:, 1]

# Reason code: which feature drove the prediction most for this row
def reason_code_fn(row):
    if row["avg_position"] > 15 and row["impressions_last_30d"] > 500:
        return "high_visibility_weak_position"
    elif row["content_age_days"] > 365:
        return "stale_content"
    elif row["ctr"] < 0.2:
        return "low_ctr"
    else:
        return "general_decline_pattern"

model_df["reason_code"] = model_df.apply(reason_code_fn, axis=1)

# Action label based on model probability
def action_label_fn(prob):
    if prob > 0.7:
        return "refresh_now"
    elif prob > 0.4:
        return "monitor"
    else:
        return "no_action"

model_df["action_label"] = model_df["decline_prob"].apply(action_label_fn)

ranked_queue = model_df.sort_values("decline_prob", ascending=False).reset_index(drop=True)

print(ranked_queue["action_label"].value_counts())
print("\nTop 10:")
print(ranked_queue.head(10)[["content_id", "content_age_days", "avg_position",
                               "impressions_last_30d", "decline_prob", "reason_code", "action_label"]])

action_label
monitor        19646
no_action       7246
refresh_now     3108
Name: count, dtype: int64

Top 10:
             content_id  content_age_days  avg_position  impressions_last_30d  \
0  content_88a2e9238054                95           1.0                    57   
1  content_b012f02fa8f1                95           0.9                    81   
2  content_637107baa450               106           0.7                   173   
3  content_8229343c9014               153           2.9                    61   
4  content_96306a965a56               165           0.6                     8   
5  content_0be51c9e6cbd               106           0.7                   273   
6  content_6b09c696507d               106           0.8                   283   
7  content_d8020ed269ef               148           1.2                   119   
8  content_6752b50ee577               131           2.9                    53   
9  content_2a57f2016a14               131           0.6                     9  

## **Ranked queue:**
3,108 pages flagged refresh_now, 19,646 monitor, 7,246 no_action
(out of ~30,000 total). The top-10 highest-priority pages share a striking
pattern: excellent average position (0.6-2.9, i.e. rank #1-3) but low CTR —
reason code "low_ctr" across the board. This means these pages already rank
at the top of search results but aren't converting that visibility into
clicks, likely due to weak titles, meta descriptions, or search snippets
rather than a ranking problem. This is a genuinely actionable, specific
signal — very different from a generic "old content" flag, and it's a
pattern the ML-07 baseline rule (which only used age + position) could
never have surfaced, since it didn't include CTR at all.

Trust note: unlike ML-07's baseline queue (where top picks were noisy,
1-2 impression outliers), this model's top-10 picks all have solid traffic
volume (57-283 impressions in the last 30 days), making the priority ranking
more reliable and trustworthy for a human reviewer.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Sanity check: how many pages in ranked queue have low reliability (low impressions)?
low_traffic_flagged = ranked_queue[
    (ranked_queue["action_label"] == "refresh_now") &
    (ranked_queue["impressions_last_30d"] < 10)
]
print(f"refresh_now picks with <10 impressions: {len(low_traffic_flagged)} / {(ranked_queue['action_label']=='refresh_now').sum()}")

refresh_now picks with <10 impressions: 351 / 3108


## **2. Intended Use and Limits**

**Who should use this:** Content editors deciding which pages to prioritize for refresh this month. Output is a *decision-support ranking*, not an automated action.

**Valid for:**
- Pages similar to training data: single month (2026-03) of GSC-style performance signals
- Directional prioritization (what to look at first), not certainty about individual pages

**Not valid for:**
- Real-time or per-page automated refresh triggers
- Pages with very low impressions (model trained on aggregate patterns, unreliable on noisy low-traffic pages — same caveat as ML-07 baseline)
- Claims about *why* Google ranks a page a certain way — model predicts a decline label, it does not model Google's algorithm
- Seasonal/trend shifts beyond the training window (only 1 month of data, no time-based validation possible)

**Breaks down when:**
- Content type differs significantly from training distribution (e.g. new page categories)
- Underlying search behavior shifts (algorithm updates, seasonality) — model would need **retraining**

## **3. Human Review and No-Go List**

**Always human-reviewed before action:**
- Top 20 `refresh_now` picks each cycle — spot-check against actual page content before editor assigns work
- Any page flagged `refresh_now` with `impressions_last_30d` < 10 — low-traffic noise risk (see Section 2 check)
- Pages where `reason_code` = `low_ctr` but `avg_position` is excellent (rank 1-3) — may be a snippet/meta issue, not a content-quality issue; needs manual diagnosis, not blind rewrite

**No-go — never automate:**
- Auto-publishing content changes based on model output
- Deprioritizing/removing pages flagged `no_action` without manual confirmation (false negatives exist — see w05 Section 4, model under-flags high-traffic decliners)
- Using this model's output as the sole justification in client-facing reports — always pair with human-verified traffic data
- Treating `decline_prob` as a certainty score — it's a ranking signal, not a probability guarantee (class imbalance + single-month training)

**Escalate to human judgment when:**
- A page's `reason_code` conflicts with editor's known context (e.g. page was intentionally deprioritized)
- Model confidence is borderline (prob 0.4–0.5, near the monitor/no_action line)

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

## 4. Monitoring and Retrain Triggers

**Signals that recommendations have gone stale:**
- New month of GSC data available — model trained on single month (2026-03), predictions should refresh monthly at minimum
- Editor feedback loop: if manually-reviewed `refresh_now` picks are frequently overturned (page wasn't actually declining), flag for retrain
- Distribution shift: if incoming pages' feature ranges (age, impressions, position) fall well outside training data range, model reliability drops

**Retrain triggers:**
- Every new monthly data pull (minimum cadence given only 1 month currently used)
- If false negative rate on high-traffic pages (known w05 weakness) shows up repeatedly in editor reviews
- Any major site-wide event — migration, algorithm update fallout, large content purge — that could shift baseline patterns

**What to monitor over time:**
- AUC on new held-out data (compare to current 0.760 benchmark)
- Ratio of refresh_now/monitor/no_action — big swings without a real cause = investigate
- % of refresh_now picks with <10 impressions (currently 11.3%) — rising trend = data quality issue

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*


Save the ranked queue and supporting artifacts so the capstone paper can reference them directly, without re-running the notebook.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
os.makedirs("work/outputs", exist_ok=True)

# 1. Full ranked queue (all pages, all columns needed for paper tables)
export_cols = [
    "content_id", "content_age_days", "avg_position", "ctr",
    "impressions_last_30d", "decline_prob", "reason_code", "action_label"
]
ranked_queue[export_cols].to_csv("work/outputs/w07_ranked_action_queue.csv", index=False)

# 2. Top-10 preview (for paper's example table)
ranked_queue[export_cols].head(10).to_csv("work/outputs/w07_top10_preview.csv", index=False)

# 3. Summary stats (counts by action_label + reason_code)
summary = ranked_queue.groupby(["action_label", "reason_code"]).size().reset_index(name="count")
summary.to_csv("work/outputs/w07_action_summary.csv", index=False)

print("Exported:")
print("- work/outputs/w07_ranked_action_queue.csv")
print("- work/outputs/w07_top10_preview.csv")
print("- work/outputs/w07_action_summary.csv")
print(summary)

Exported:
- work/outputs/w07_ranked_action_queue.csv
- work/outputs/w07_top10_preview.csv
- work/outputs/w07_action_summary.csv
   action_label                    reason_code  count
0       monitor        general_decline_pattern   4229
1       monitor  high_visibility_weak_position   2163
2       monitor                        low_ctr   9807
3       monitor                  stale_content   3447
4     no_action        general_decline_pattern   1654
5     no_action  high_visibility_weak_position    789
6     no_action                        low_ctr   2416
7     no_action                  stale_content   2387
8   refresh_now        general_decline_pattern   1272
9   refresh_now  high_visibility_weak_position    153
10  refresh_now                        low_ctr   1663
11  refresh_now                  stale_content     20


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.